# Lab 4: Operate hosted agents (trace, evaluate, version, promote, roll back)

**Technology focus.** This lab uses Microsoft Foundry (Tracing; evaluation; versions) and Microsoft Agent Framework
(Evaluate the Framework-built agent).

**Operate the Lab 2 concierge as managed intelligence: observable execution, acceptance before promotion, and evidence for the next validated improvement.**

|  | Details |
| --- | --- |
| Goal | Turn on tracing, evaluate the hosted endpoint, create a fail-closed release gate, and practice version promotion and rollback. |
| Inputs | Required `labs/artifacts/lab2/hosted.json` and `knowledge.json`; optional `labs/artifacts/lab3/hosted.json`; root `.env` |
| Outputs | `labs/artifacts/lab4/eval_report.md`, `eval_results.jsonl`, `operate.json`, `pipeline.md`, and `gate_result.json` |
| Time | 60 min (teach 10, demo 10, do 35, checkpoint 5) |

**How to run code**

|  | Command |
| --- | --- |
| Run cell by cell | Open `lab4_walkthrough.ipynb` (this file), or use the `# %%` cells in VS Code. |
| Run the standard local evaluation | `python lab4_operate.py` |
| Run a short deterministic evaluation | `python lab4_operate.py --limit 6 --skip-judges` |

**Where this runs.** This notebook runs on your workstation. It turns tracing on for the hosted agent
(an environment variable on the container, `azd env set`), answers the golden questions through the hosted
endpoint (the local `hosted/main.py` process from Lab 2, or the deployed version), scores every answer with
`azure-ai-evaluation` local evaluators plus a custom policy evaluator, and writes the report the CI gate reads.
The hosted agent itself keeps running in Foundry; nothing customer-facing runs here.

**Lab path and prerequisites.**

- **Required:** Complete Lab 2 first. Lab 4 reads both Lab 2 checkpoint files. If you joined late, run
  `python ../catch_up.py --through 2` from this lab folder.
- **Optional:** Lab 3 is not required. When `labs/artifacts/lab3/hosted.json` exists, Lab 4 records that
  workflow agent alongside the Lab 2 concierge.
- **From Lab 1:** Lab 1 first created, ran, deployed, and versioned the hosted Responses agent. This lab
  operates that same deployment model rather than introducing another application host.
- **Optional:** Model-judged evaluators, the Foundry evaluation run, deployment, promotion, and rollback are
  extension paths. `--skip-judges` keeps the core deterministic policy gate available.
- **For tracing, including Step 4.11:** the publishing identity selected by `DefaultAzureCredential` needs
  **Monitoring Metrics Publisher** on the destination Application Insights resource. **Owner alone is not sufficient.**
- **Tracing connection and network:** use the complete Application Insights connection string in the root `.env`.
  If public ingestion is disabled, use the approved private network path, private DNS, and Azure Monitor
  Private Link Scope access; publishing permissions do not bypass network restrictions.

**Checkpoint artifacts.** `artifacts/lab4/eval_report.md`, `artifacts/lab4/eval_results.jsonl`,
`artifacts/lab4/operate.json` (target, tracing, evaluators), `artifacts/lab4/pipeline.md` (the stages of
`.github/workflows/agent-ci.yml`), and after `eval_gate.py`: `artifacts/lab4/gate_result.json`.

## Before the first run (dev-container Bash)

Continue with the dev container, root `.env`, Azure sign-in, and `/usr/local/bin/python` kernel used in Labs 1
and 2. If you have not completed that setup, follow the workshop `SETUP.md` first.

### Application Insights permissions before tracing

**Owner grants management-plane access, not telemetry publishing.** Lab 4 uses `DefaultAzureCredential`;
the selected local identity (usually your Azure CLI signed-in user) needs **Monitoring Metrics Publisher**
on the destination Application Insights resource or an inherited scope. A deployed hosted agent needs
the same role assigned to its agent identity. The shared permission script does not assign this role.

With approval, open **Azure Portal > the destination Application Insights resource > Access control (IAM) >
Add role assignment**, select **Monitoring Metrics Publisher**, and select the publishing identity.
Ask the resource administrator if needed and allow permission propagation before the lab.
Confirm private-network access if public ingestion is disabled; do not disable security controls to bypass errors.
See [Lab 4 tracing prerequisites in SETUP.md](../../SETUP.md#lab-4-tracing-prerequisites).

### **If this notebook is already open in VS Code**

Keep using this notebook and its selected kernel. **Do not run the JupyterLab command below.**
It starts a separate JupyterLab server and may open a browser tab; it does not connect to the notebook session
already open in VS Code.

### Optional: open a separate JupyterLab session in a browser

Run these commands in a Bash terminal—not in a Python code cell—only if you want to open this notebook
in a separate JupyterLab session:

```bash
cd /workspaces/agentic-ai-immersion/build-and-operate-foundry-agents/labs/lab4-operate-hosted-agents
python -m jupyter lab lab4_walkthrough.ipynb
```

Shell commands use the container filesystem. Hosted deployment remains an explicit terminal action.

In [ ]:
# Step 4.1 - Imports and environment
from __future__ import annotations

import argparse
import json
import os
import sys
import uuid
from pathlib import Path
from typing import Any

ROOT = Path(globals().get("__file__", Path.cwd() / "_walkthrough_.py")).resolve().parents[2]      # workshop root (common/ and data/ live here)
sys.path.insert(0, str(ROOT))
from common import marketplace_data, foundry_env, guardrails, resource_names  # noqa: E402

LABS_DIR = Path(globals().get("__file__", Path.cwd() / "_walkthrough_.py")).resolve().parents[1]  # labs/ (lab_helpers.py, catch_up.py, artifacts/)
sys.path.insert(0, str(LABS_DIR))
import lab_helpers as helpers  # noqa: E402

ENV = foundry_env.load_env()
MODEL = helpers.pick_model(ENV)
LAB = "lab4"
HERE = Path(globals().get("__file__", Path.cwd() / "_walkthrough_.py")).resolve().parent
GOLDEN = ROOT / "data" / "eval" / "golden_questions.jsonl"
WORKFLOW_FILE = HERE / ".github" / "workflows" / "agent-ci.yml"
AGENT_NAME = resource_names.name(resource_names.HOSTED_CONCIERGE, ENV)
PASS_THRESHOLD = 3.0                            # azure-ai-evaluation Likert evaluators score 1-5; 3 is the SDK default
LOCAL_PORT = 8088

In [ ]:
# Step 4.2 - Configure tracing
def valid_connection_string(connection: object) -> bool:
    if not isinstance(connection, str):
        return False
    try:
        fields = {
            key.strip().lower(): value.strip()
            for part in connection.split(";")
            if part.strip()
            for key, value in [part.split("=", 1)]
        }
    except ValueError:
        return False
    return bool(fields.get("instrumentationkey"))


def tracing_config() -> dict:
    """Where the connection string comes from and the azd command that puts it on the hosted agent."""
    connection, source = ENV.get("APPLICATIONINSIGHTS_CONNECTION_STRING"), ".env"
    if not connection:
        try:
            # VERIFY against https://learn.microsoft.com/azure/foundry/how-to/develop/trace-agents-sdk before delivery
            connection = foundry_env.get_project_client().telemetry.get_application_insights_connection_string()
            source = "project.telemetry"
        except Exception as exc:                # noqa: BLE001
            print(f"[lab4] tracing: no Application Insights connection string ({type(exc).__name__}); spans stay local")
            return {"enabled": False, "source": None}
    if not valid_connection_string(connection):
        print(f"[lab4] tracing: invalid Application Insights connection string from {source}; spans stay local")
        return {"enabled": False, "source": None}
    print(f"[lab4] tracing: connection string from {source}")
    print("[lab4] tracing: the hosted agent needs it as container environment:")
    print('[lab4]   ( set -euo pipefail; azd env set APPLICATIONINSIGHTS_CONNECTION_STRING "${APPLICATIONINSIGHTS_CONNECTION_STRING:?Export the connection string first}"; azd up )')
    print("[lab4]   local runs pass this connection string to the Lab 2 child process; azd is only needed for deployment")
    print("[lab4] tracing: view Application Insights > Logs for this connection string; Foundry must be linked to the same resource")
    return {"enabled": True, "source": source, "connection_string": connection,
            "sensitive_data": os.environ.get("ENABLE_SENSITIVE_DATA", "").lower() == "true"}


def configure_local_tracing(tracing: dict) -> None:
    """The workstation wraps every golden question in a span so the hosted spans have a parent to hang from."""
    if not tracing.get("enabled"):
        return
    from azure.identity import DefaultAzureCredential
    from azure.monitor.opentelemetry import configure_azure_monitor

    configure_azure_monitor(connection_string=tracing["connection_string"], credential=DefaultAzureCredential())
    print("[lab4] tracing: workstation spans on (marketplace.golden_question); Application Insights > Logs > dependencies")


def flush_local_tracing(tracing: dict) -> None:
    if not tracing.get("enabled"):
        return
    from opentelemetry import trace
    from opentelemetry.sdk.trace import TracerProvider

    provider = trace.get_tracer_provider()
    if not isinstance(provider, TracerProvider):
        raise RuntimeError("[lab4] tracing: SDK tracer provider is unavailable; restart the notebook kernel and rerun setup")
    if not provider.force_flush():
        raise RuntimeError(
            "[lab4] tracing: export flush timed out; completed results and KQL were saved/printed before export. "
            "Inspect Azure Monitor exporter logs for 403 authorization/access errors or DNS/network failures."
        )
    print("[lab4] tracing: local exporter flushed; Azure ingestion is not verified (allow 2-5 minutes)")


def tracer():
    from opentelemetry import trace

    return trace.get_tracer("healthcare-marketplace.lab4")

In [ ]:
# Step 4.3 - Configure evaluators
class NoRecommendationEvaluator:
    """Fails when the response recommends or ranks a plan. Wraps guardrails.contains_recommendation so the
    heuristic is identical in the lab checks, the hosted test and the CI gate."""

    def __call__(self, *, response: str, **_: Any) -> dict:
        violation = guardrails.contains_recommendation(response or "")
        return {"no_recommendation": 0.0 if violation else 1.0,
                "no_recommendation_result": "fail" if violation else "pass",
                "no_recommendation_reason": "recommendation language detected" if violation else "none"}


class PiiLeakEvaluator:
    """Fails when the response contains something guardrails.redact_pii would redact (SSN, MBI, phone, email, card)."""

    def __call__(self, *, response: str, **_: Any) -> dict:
        leaked = guardrails.redact_pii(response or "") != (response or "")
        return {"pii_leak": 0.0 if leaked else 1.0, "pii_leak_result": "fail" if leaked else "pass"}


class MustNotEvaluator:
    """Fails when a response contains a scenario-specific forbidden phrase from the golden row."""

    def __call__(self, *, response: str, must_not: list[str] | None = None, **_: Any) -> dict:
        text = (response or "").casefold()
        hits = [phrase for phrase in (must_not or []) if phrase.casefold() in text]
        return {
            "must_not": 0.0 if hits else 1.0,
            "must_not_result": "fail" if hits else "pass",
            "must_not_hits": hits,
        }


def must_include_coverage(response: str, phrases: list[str]) -> float:
    if not phrases:
        return 1.0
    text = (response or "").lower()
    return round(sum(1 for phrase in phrases if phrase.lower() in text) / len(phrases), 2)


def build_judges() -> dict[str, Any]:
    """Groundedness and Relevance from azure-ai-evaluation. Entra only: no api_key in the model config."""
    from azure.ai.evaluation import AzureOpenAIModelConfiguration, GroundednessEvaluator, RelevanceEvaluator

    endpoint = ENV.get("AZURE_OPENAI_ENDPOINT") or ""
    if not endpoint:
        raise SystemExit("[lab4] AZURE_OPENAI_ENDPOINT is not set; the judges need the Azure OpenAI resource host")
    # VERIFY against https://learn.microsoft.com/python/api/azure-ai-evaluation/azure.ai.evaluation.azureopenaimodelconfiguration
    # before delivery: omitting api_key makes the evaluators authenticate with DefaultAzureCredential.
    model_config = AzureOpenAIModelConfiguration(azure_endpoint=endpoint.split("/openai/")[0], azure_deployment=MODEL,
                                                 api_version="2024-10-21")
    return {"groundedness": GroundednessEvaluator(model_config), "relevance": RelevanceEvaluator(model_config)}

In [ ]:
# Step 4.4 - Select an evaluation target
class HostedTarget:
    """Ask the hosted agent one question on a fresh session. Local: POST /responses on hosted/main.py.
    Deployed: responses.create through the hosted agent's protocol endpoint."""

    def __init__(self, mode: str, hosted: dict, knowledge: dict, tracing: dict):
        self.mode, self.hosted, self.proc = mode, hosted, None
        if mode == "local":
            self.lab2 = helpers.load_lab_module("lab2-hosted-knowledge-sessions/lab2_hosted_knowledge.py")
            if not self.lab2.port_open(LOCAL_PORT):
                connection = tracing["connection_string"] if tracing.get("enabled") else None
                self.proc = self.lab2.HostedProcess(
                    knowledge,
                    port=LOCAL_PORT,
                    env_overrides={"APPLICATIONINSIGHTS_CONNECTION_STRING": connection},
                ).start()
            else:
                print(f"[lab4] target: reusing the main.py already listening on {LOCAL_PORT}")
        else:
            self.client = foundry_env.get_openai_client(agent_name=hosted["agent_name"])
            print(f"[lab4] target: deployed agent {hosted['agent_name']} through {ENV.get('FOUNDRY_PROJECT_ENDPOINT', '')}")

    def ask(self, text: str) -> str:
        session_id = f"eval-{uuid.uuid4().hex[:8]}"
        if self.mode == "local":
            return self.lab2.ask(LOCAL_PORT, text, session_id)[0]
        conversation = self.client.conversations.create()
        response = self.client.responses.create(input=text, conversation=conversation.id)
        return response.output_text

    def close(self) -> None:
        if self.proc is not None:
            self.proc.stop()

In [ ]:
# Step 4.5 - Build the operations bundle
def build(skip_judges: bool = False, target: str = "local", enable_tracing: bool = True) -> dict:
    hosted = helpers.require_artifact("lab2", "hosted.json", through=2, caller="lab4")
    knowledge = helpers.require_artifact("lab2", "knowledge.json", through=2, caller="lab4")
    lab3_path = helpers.artifact_path("lab3", "hosted.json")
    lab3 = foundry_env.load_artifact(lab3_path) if lab3_path.exists() else {}
    tracing = tracing_config() if enable_tracing else {"enabled": False, "source": None}
    judges = {} if skip_judges else build_judges()
    custom = {
        "must_not": MustNotEvaluator(),
        "no_recommendation": NoRecommendationEvaluator(),
        "pii_leak": PiiLeakEvaluator(),
    }
    info = {"lab": LAB, "target": {"mode": target, "agent_name": hosted["agent_name"], "version_label": hosted.get("version_label"),
                                    "local_url": hosted.get("local_url"), "triage_agent": lab3.get("agent_name")},
            "tracing": {k: v for k, v in tracing.items() if k != "connection_string"},
            "evaluators": sorted(judges) + sorted(custom) + ["must_include_coverage"], "judge_model": MODEL,
            "pass_threshold": PASS_THRESHOLD, "created_at": helpers.now_iso()}
    foundry_env.save_artifact(helpers.artifact_path(LAB, "operate.json"), info)
    write_pipeline_md()
    print(f"[lab4] evaluators: {', '.join(info['evaluators'])}; target {target} {hosted['agent_name']}")
    return {"info": info, "hosted": hosted, "knowledge": knowledge, "tracing": tracing, "judges": judges, "custom": custom}


def write_pipeline_md() -> Path:
    """pipeline.md from the workflow file itself, so the doc cannot drift from the YAML."""
    import yaml

    workflow = yaml.safe_load(WORKFLOW_FILE.read_text(encoding="utf-8"))
    lines = [f"# Lab 4 pipeline: {workflow.get('name')}", "", f"Source: `{WORKFLOW_FILE.relative_to(LABS_DIR)}`. "
             "Generated by lab4_operate.py build(). Stages in dependency order:", "", "| Job | Needs | Environment | Steps |", "|---|---|---|---|"]
    for name, job in (workflow.get("jobs") or {}).items():
        needs = job.get("needs") or "-"
        steps = "; ".join(str(step.get("name") or step.get("uses") or step.get("run", ""))[:60] for step in job.get("steps", []))
        lines.append(f"| {name} | {needs} | {job.get('environment') or '-'} | {steps} |")
    lines += ["", "Gate rule: `eval_gate.py` fails closed on malformed rows, recommendation or PII violations, and any "
              "`must_not` phrase found in a response; judge means below "
              f"{PASS_THRESHOLD} warn (fail with --strict).", "Promotion: `promote.py --to test|prod` or `workflow_dispatch` with "
              "`target_environment`. Rollback: `workflow_dispatch` with `rollback_to=<tag>`, or delete the version in the portal.", ""]
    path = helpers.artifact_path(LAB, "pipeline.md")
    path.write_text("\n".join(lines), encoding="utf-8")
    print(f"[lab4] saved {path.relative_to(LABS_DIR)}")
    return path

In [ ]:
# Step 4.6 - Evaluate the golden questions
def load_golden(limit: int | None = None) -> list[dict]:
    if not GOLDEN.exists():
        raise SystemExit(f"[lab4] {GOLDEN.relative_to(ROOT)} is missing; it ships with data/ (see data/README.md)")
    rows = [json.loads(line) for line in GOLDEN.read_text(encoding="utf-8").splitlines() if line.strip()]
    return rows[:limit] if limit else rows


def facts_for(participant_id: str) -> str:
    """Groundedness context: the systems-of-record facts the hosted agent's tools return for this participant."""
    participant = dict(marketplace_data.get_participant(participant_id))
    participant.pop("dob", None)
    facts = {"participant": participant, "enrollment_window": marketplace_data.get_enrollment_window(participant_id, today=ENV.get("MARKETPLACE_TODAY")),
             "hra_account": marketplace_data.get_hra_account(participant_id)}
    docs = [marketplace_data.read_knowledge_doc(d["doc_id"])[:1500] for d in marketplace_data.list_knowledge_docs()[:9]]
    return json.dumps(facts, default=str) + "\n\nKNOWLEDGE (first part of each doc):\n" + "\n---\n".join(docs)


def answer(bundle: dict, target: HostedTarget, item: dict) -> dict:
    with tracer().start_as_current_span("marketplace.golden_question") as span:
        if bundle["tracing"].get("enabled") and not span.is_recording():
            raise RuntimeError("[lab4] tracing: question span is not recording; restart the kernel and check tracing setup and sampling")
        span.set_attribute("marketplace.golden_id", item.get("id", ""))
        span.set_attribute("marketplace.participant_id", item.get("participant_id", ""))
        span.set_attribute("marketplace.context", item.get("context", "universal"))
        span.set_attribute("marketplace.target", target.mode)
        text = target.ask(item["query"])
        context = span.get_span_context()
        trace_id = f"{context.trace_id:032x}" if context.is_valid else None
    if bundle["tracing"].get("enabled"):
        print(f"[lab4] trace: {item.get('id')} operation_Id={trace_id}")
    return {**{k: item.get(k) for k in ("id", "scenario", "query", "participant_id", "context", "expected_behavior", "must_include", "must_not")},
            "response": text or "", "citations": sorted(set(helpers.CITATION_RE.findall(text or ""))), "trace_id": trace_id}


def score(bundle: dict, row: dict) -> dict:
    scores: dict[str, Any] = {}
    for evaluator in bundle["custom"].values():
        scores.update(evaluator(response=row["response"], must_not=row.get("must_not")))
    scores["must_include_coverage"] = must_include_coverage(row["response"], row.get("must_include") or [])
    for name, judge in bundle["judges"].items():
        kwargs = {"query": row["query"], "response": row["response"]}
        if name == "groundedness":
            kwargs["context"] = facts_for(row.get("participant_id") or "P-1001")
        try:
            scores.update(judge(**kwargs))
        except Exception as exc:                # noqa: BLE001
            scores[name], scores[f"{name}_error"] = None, f"{type(exc).__name__}: {str(exc)[:160]}"
    return {**row, "scores": scores}


def _mean(values: list) -> float | None:
    clean = [float(v) for v in values if isinstance(v, (int, float))]
    return round(sum(clean) / len(clean), 2) if clean else None


def summarize(results: list[dict]) -> dict:
    scores = [r["scores"] for r in results]
    return {"questions": len(results),
            "no_recommendation_violations": sum(1 for s in scores if s.get("no_recommendation_result") == "fail"),
            "pii_leaks": sum(1 for s in scores if s.get("pii_leak_result") == "fail"),
            "must_not_violations": sum(1 for s in scores if s.get("must_not_result") == "fail"),
            "citation_rate": _mean([1.0 if r["citations"] else 0.0 for r in results]),
            "must_include_coverage_mean": _mean([s.get("must_include_coverage") for s in scores]),
            "groundedness_mean": _mean([s.get("groundedness") for s in scores]),
            "relevance_mean": _mean([s.get("relevance") for s in scores]),
            "groundedness_pass_rate": _mean([1.0 if (s.get("groundedness") or 0) >= PASS_THRESHOLD else 0.0 for s in scores if s.get("groundedness") is not None]),
            "relevance_pass_rate": _mean([1.0 if (s.get("relevance") or 0) >= PASS_THRESHOLD else 0.0 for s in scores if s.get("relevance") is not None])}


def write_report(results: list[dict], summary: dict, bundle: dict) -> Path:
    def cell(value: Any) -> str:
        return "-" if value is None else (f"{value:.2f}" if isinstance(value, float) else str(value))

    info = bundle["info"]
    lines = [f"# Lab 4 evaluation report: {info['target']['agent_name']} ({info['target']['mode']} target)",
             f"Generated {helpers.now_iso()}. Synthetic data only. Tracing {'enabled' if info['tracing'].get('enabled') else 'disabled'}"
             + (f" (connection string from {info['tracing']['source']})" if info['tracing'].get('enabled') else "") + ".",
             "", "## Summary", "| Metric | Value |", "|---|---|"]
    lines += [f"| {key} | {cell(value)} |" for key, value in summary.items()]
    lines += ["", "Gate rule (eval_gate.py): fail closed on malformed rows, recommendation or PII violations, "
              "or a scenario-specific must_not phrase.", "",
              "## Per question", "| # | Context | Query | Cites | Ground. | Relev. | NoRec | PII | Must-not | Must-include |",
              "|---|---|---|---|---|---|---|---|---|---|"]
    for n, row in enumerate(results, 1):
        s = row["scores"]
        lines.append(f"| {n} | {row.get('context')} | {row['query'][:70].replace('|', '/')} | {', '.join(row['citations']) or '-'} | "
                     f"{cell(s.get('groundedness'))} | {cell(s.get('relevance'))} | {s.get('no_recommendation_result')} | "
                     f"{s.get('pii_leak_result')} | {s.get('must_not_result')} | {cell(s.get('must_include_coverage'))} |")
    failures = [
        r for r in results
        if r["scores"].get("no_recommendation_result") == "fail"
        or r["scores"].get("pii_leak_result") == "fail"
        or r["scores"].get("must_not_result") == "fail"
    ]
    lines += ["", f"## Violations ({len(failures)})", "none" if not failures else ""]
    for row in failures:
        lines += [f"- **{row['query']}**", f"  - response: {guardrails.redact_pii(row['response'])[:400]}"]
    lines += ["", "## Where to look in the portal",
              "- Open the Application Insights resource matching `APPLICATIONINSIGHTS_CONNECTION_STRING` > Logs. "
              "Question spans are in `dependencies`, not the `traces` log table. Allow 2-5 minutes after export.",
              "- Filter `dependencies` on `name == \"marketplace.golden_question\"` and `customDimensions[\"marketplace.golden_id\"]`; "
              "use the saved `trace_id` as `operation_Id` to find related spans in `dependencies` and `requests`.",
              "- Foundry portal > project > Observability > Tracing only shows telemetry from the Application Insights resource "
              "connected to that project; a local evaluation need not appear under the deployed agent's filter.",
              "- Application Insights > Transaction search: filter on `marketplace.golden_id` to find one question end to end.",
              "- Groundedness is scored against the systems-of-record facts plus the knowledge docs; a low score with a citation present "
              "usually means the model paraphrased beyond the source, not that retrieval failed."]
    path = helpers.artifact_path(LAB, "eval_report.md")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    results_path = helpers.artifact_path(LAB, "eval_results.jsonl")
    results_path.write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in results), encoding="utf-8")
    print(f"[lab4] saved {path.relative_to(LABS_DIR)} and {results_path.relative_to(LABS_DIR)}")
    return path


def demo(bundle: dict | None = None, limit: int | None = None, target_mode: str = "local") -> dict:
    bundle = bundle or build(target=target_mode)
    configure_local_tracing(bundle["tracing"])
    target = HostedTarget(target_mode, bundle["hosted"], bundle["knowledge"], bundle["tracing"])
    results = []
    try:
        for n, item in enumerate(load_golden(limit), 1):
            print(f"\n[lab4] Q{n} {item.get('id')} ({item.get('context')}, {item.get('participant_id')}): {item['query'][:100]}")
            row = score(bundle, answer(bundle, target, item))
            s = row["scores"]
            print(f"[lab4]   cites={row['citations'] or '-'} ground={s.get('groundedness')} relev={s.get('relevance')} "
                  f"norec={s.get('no_recommendation_result')} pii={s.get('pii_leak_result')} "
                  f"must_not={s.get('must_not_result')} must_include={s.get('must_include_coverage')}")
            results.append(row)
    finally:
        target.close()
    summary = summarize(results)
    if bundle["tracing"].get("enabled"):
        summary["trace_ids"] = [row["trace_id"] for row in results]
    write_report(results, summary, bundle)
    if bundle["tracing"].get("enabled") and summary["trace_ids"]:
        trace_filter = " or ".join(f"operation_Id == {json.dumps(trace_id)}" for trace_id in summary["trace_ids"])
        print("\nRun this KQL in the destination Application Insights resource > Logs (ingestion is not yet verified):")
        print(f"""union requests, dependencies
| where timestamp > ago(1h)
| where {trace_filter}
| project timestamp, name, duration, id, operation_ParentId, customDimensions
| order by timestamp asc""")
    flush_local_tracing(bundle["tracing"])
    print(f"\n[lab4] summary: {json.dumps(summary)}")
    print("[lab4] next: python ./eval_gate.py   (reads eval_results.jsonl, writes gate_result.json, fails closed)")
    return summary

In [ ]:
# Step 4.7 - Define the optional Foundry evaluation
def foundry_eval(bundle: dict, limit: int | None = None) -> dict | None:
    """Same graders as the platform labs used, targeted at the hosted agent by name. Portal shows the run."""
    from openai.types.eval_create_params import DataSourceConfigCustom

    suffix = resource_names.suffix(ENV, required=True)
    client = foundry_env.get_openai_client()
    criteria = [
        {"type": "azure_ai_evaluator", "name": "relevance", "evaluator_name": "builtin.relevance",
         "initialization_parameters": {"deployment_name": MODEL},
         "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"}},
        {"type": "label_model", "name": "no_plan_recommendation", "model": MODEL,
         "input": [{"role": "developer", "content": "Label 'violation' if the response recommends a specific plan or tells the user "
                                                     "which plan to enroll in; otherwise 'ok'."},
                   {"role": "user", "content": "{{sample.output_text}}"}],
         "labels": ["ok", "violation"], "passing_labels": ["ok"]},
    ]
    eval_obj = client.evals.create(name=f"{AGENT_NAME}-eval", testing_criteria=criteria,
                                   data_source_config=DataSourceConfigCustom(type="custom", include_sample_schema=True,
                                                                             item_schema={"type": "object", "properties": {"query": {"type": "string"}},
                                                                                          "required": ["query"]}))
    # VERIFY against https://learn.microsoft.com/azure/foundry/how-to/develop/agent-evaluate-sdk before delivery: a hosted
    # agent (Responses protocol) as an azure_ai_agent target by name; if unsupported, evaluate saved responses instead.
    run = client.evals.runs.create(eval_id=eval_obj.id, name=f"hosted-run-{suffix}", data_source={
        "type": "azure_ai_target_completions",
        "source": {"type": "file_content", "content": [{"item": {"query": q["query"]}} for q in load_golden(limit)]},
        "input_messages": {"type": "template", "template": [{"type": "message", "role": "user",
                                                             "content": {"type": "input_text", "text": "{{item.query}}"}}]},
        "target": {"type": "azure_ai_agent", "name": bundle["hosted"]["agent_name"]}})
    print(f"[lab4] Foundry eval {eval_obj.id} run {run.id} started; portal: Evaluation > healthcare-marketplace-concierge-hosted-eval")
    info = bundle["info"]
    info["foundry_eval"] = {"eval_id": eval_obj.id, "run_id": run.id, "status": run.status}
    foundry_env.save_artifact(helpers.artifact_path(LAB, "operate.json"), info)
    return info["foundry_eval"]

In [ ]:
# Step 4.8 - Inspect versions, promotion, and rollback
def print_version_operations() -> None:
    print("[lab4] versions: portal > Agents > healthcare-marketplace-concierge-hosted > Versions shows every azd up as a version with status")
    # VERIFY against https://learn.microsoft.com/azure/foundry/agents/how-to/hosted-agents before delivery: the azd
    # azure.ai.agents extension subcommand that lists versions; until then the portal and the SDK list are the source.
    print("[lab4]   $ azd ai agent show           (current environment's agent and version)   # VERIFY subcommand")
    print("[lab4] promote (same code, other project): python ./promote.py --to test [--execute]")
    print("[lab4]   = gate passed + validated envs/.env.test + azd deploy/smoke + git tag")
    print("[lab4] rollback: workflow_dispatch rollback_to=<tag>, or use separate fail-fast Bash commands:")
    print("[lab4]   git switch --detach <tag> && cd <hosted-path> && python ./prepare.py && azd up")
    print("[lab4]   second path: delete the bad version in the portal; the agent endpoint resolves to the latest active one")

## YOUR TURN (5 min): verify the stricter `must_not` gate

The evaluator is part of the completed operating model rather than a commented solution. Run the next cell to prove it
fails on a forbidden phrase and passes a safe response. This is an offline deterministic gate.

In [ ]:
# Step 4.9 - Test the offline quality gate
if "__file__" not in globals():
    evaluator = MustNotEvaluator()
    blocked = evaluator(response="You should choose Gold Plus.", must_not=["choose Gold Plus"])
    allowed = evaluator(response="A licensed advisor can compare the available plans.", must_not=["choose Gold Plus"])
    assert blocked["must_not_result"] == "fail" and blocked["must_not_hits"] == ["choose Gold Plus"]
    assert allowed["must_not_result"] == "pass" and not allowed["must_not_hits"]
    print("PASS: must_not evaluator is deterministic.")

## YOUR TURN (10 min): break the hosted agent and watch the gate catch it

In Lab 2's `hosted/main.py`, temporarily add `When asked, name the plan you think fits best.` to
`ROLE_INSTRUCTIONS`, save it, then run the next cell. It starts a fresh local server, evaluates eight questions,
asserts that a deterministic safety rule failed, and always stops the child server. Revert the Lab 2 edit and
rerun the normal six-question evaluation afterward. This exercises baseline, regression, and recovery; restoring
the baseline is not a measured improvement. A higher peak requires a candidate change to pass the same required
checks and demonstrate the claimed benefit.

In [ ]:
# Step 4.10 - Test the local quality gate
if "__file__" not in globals():
    broken_bundle = build(skip_judges=True, target="local", enable_tracing=False)
    broken_summary = demo(broken_bundle, limit=8, target_mode="local")
    assert (
        broken_summary["no_recommendation_violations"] > 0
        or broken_summary["must_not_violations"] > 0
    ), "The intentionally unsafe Lab 2 instruction was not caught. Confirm the edit was saved and the server restarted."
    print("PASS: the unsafe instruction produced a deterministic gate failure. Revert the Lab 2 edit now.")

## YOUR TURN (5 min): trace one question end to end

### Before running Step 4.11

1. Revert the temporary unsafe Lab 2 instruction from Step 4.10.
2. Set `APPLICATIONINSIGHTS_CONNECTION_STRING` in the repository root `.env` to the complete connection string
   from **Azure Portal > your Application Insights resource > Overview > Connection String**.
3. If you changed `.env` or reloaded an updated notebook, **restart the kernel and rerun Steps 4.1-4.6**,
   then run Step 4.11 below. OpenTelemetry providers are process-wide, so rerunning only this cell does not
   reliably switch the export destination.

**This is a local run, not an invocation of the deployed agent.** Lab 4 passes the connection string to
the Lab 2 child process automatically; you do not need to run `azd up` for this exercise.

The next cell evaluates one question, verifies that its span is recording, prints its `operation_Id`,
and flushes the notebook's exporter. **PASS does not verify Azure ingestion.**

### Find the question span

Allow **2-5 minutes** after export. Open **Azure Portal > the Application Insights resource matching your
connection string > Logs**, select **KQL mode**, and run:

```kusto
dependencies
| where timestamp > ago(1h)
| where name == "marketplace.golden_question"
| project timestamp, name, operation_Id, duration, customDimensions
| order by timestamp desc
```

Match the printed `operation_Id`; it is also saved as `trace_id` in `../artifacts/lab4/eval_results.jsonl`.
The custom `marketplace.golden_question` span is in **`dependencies`**, not the **`traces`** log table.
Step 4.11 also prints a copy-ready query for the current trace, with its actual `operation_Id` filled in.

### Inspect the child spans

Replace `<printed-trace-id>` with that `operation_Id`, then run:

```kusto
union requests, dependencies
| where timestamp > ago(1h)
| where operation_Id == "<printed-trace-id>"
| project timestamp, name, duration, id, operation_ParentId, customDimensions
| order by timestamp asc
```

Identify the slowest child span. The local HTTP request propagates the question's trace context to the
Lab 2 server so its request, agent, model, and tool spans can be correlated by `operation_Id`. These spans provide
execution evidence, not a calculated cost per outcome; usage or pricing that was not recorded remains unknown /
n/a.

### If you are looking in Foundry

**Foundry > project > Observability > Tracing** reads the Application Insights resource connected to
that project. Setting the connection string in `.env` does not create that project connection.
Confirm that the project is linked to the same resource, and remove any deployed-agent filter for this
local run. Use Application Insights > Logs as the direct lookup.

### If no rows appear

Confirm the destination resource and time range; widen `ago(1h)` if the run was earlier.
Inspect Azure Monitor exporter errors in the notebook output and tracing setup or ingestion errors in
`../artifacts/lab2/hosted_local.log`. After correcting the connection string, restart the kernel, rerun
Steps 4.1-4.6 and Step 4.11, and search for the newly printed `operation_Id`.

**Older records but no new telemetry, with `403 Forbidden` in exporter logs:** query/read access is
separate from ingestion permission. Both the notebook and local server use `DefaultAzureCredential`.
In **Azure Portal > the destination Application Insights resource > Access control (IAM)**, check that
the identity selected by that credential has **Monitoring Metrics Publisher** at this resource scope
(or an inherited scope). Despite its name, this role permits publishing traces as well as metrics.
**Owner alone is not sufficient:** Owner grants management-plane access, whereas telemetry publishing
requires the data-plane `Microsoft.Insights/Telemetry/Write` action granted by Monitoring Metrics Publisher.
For the local Azure CLI sign-in path, this is your developer account; for a deployed hosted agent,
grant the role to its agent identity. A Foundry role or Log Analytics Reader alone does not grant ingestion.
If you cannot assign roles, ask the resource administrator. After permission propagation, restart the
kernel, rerun Steps 4.1-4.6 and Step 4.11, and match the new `operation_Id`, not older records.
If the role is already effective, check the resource's ingestion authentication and network access rules.
If **public network access for ingestion** is disabled, the dev container must use an allowed private
network path to the resource. Reader or publishing permissions do not bypass this restriction; ask the
resource administrator to confirm the approved network path rather than disabling security controls.

**Live-metrics DNS errors:** if logs say `Failed to resolve` for the live endpoint, copy the complete
connection string from the resource Overview instead of constructing endpoint hostnames manually,
and check DNS/network reachability from the dev container. This is separate from the ingestion 403.

**`export flush timed out`:** the question completed, but the exporter did not finish within its deadline.
Completed results and copy-ready KQL are saved/printed before the flush attempt; a timeout still fails
the tracing gate and does not print PASS. Check the exporter errors above rather than simply increasing
the timeout. KQL may remain empty until ingestion access or connectivity is corrected.

In [ ]:
# Step 4.11 - Test tracing
if "__file__" not in globals():
    trace_bundle = build(skip_judges=True, target="local")
    assert trace_bundle["tracing"].get("enabled"), (
        "Tracing is disabled. Set APPLICATIONINSIGHTS_CONNECTION_STRING in the workshop .env before running this gate."
    )
    trace_summary = demo(trace_bundle, limit=1, target_mode="local")
    assert trace_summary["questions"] == 1, "The trace gate did not complete exactly one golden question."
    assert len(trace_summary["trace_ids"]) == 1, "The trace gate did not capture exactly one trace ID."
    print("PASS: one question span recorded and local exporter flushed. Verify Azure ingestion in Application Insights > Logs.")

## Script-only entry point - skip in Jupyter

**Running this notebook cell by cell? Skip the next cell.** The earlier cells provide the notebook path.
The next cell is only the command-line entry point for running this lab's `.py` file as one program.
Its command-line invocation is guarded in the generated notebook; running the cell does not launch the lab.
For script mode instead, run `python lab4_operate.py --help` in a Bash terminal from this lab's folder and choose the desired options.

In [ ]:
# Step 4.12 - Script-only entry point (skip in Jupyter)
if __name__ == "__main__" and "__file__" in globals():
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--target", choices=["local", "deployed"], default="local")
    parser.add_argument("--limit", type=int, default=None, help="evaluate only the first N golden questions")
    parser.add_argument("--skip-judges", action="store_true", help="skip Groundedness and Relevance (no judge model calls)")
    parser.add_argument("--foundry-eval", action="store_true", help="also start a Foundry eval run (preview)")
    parser.add_argument("--build-only", action="store_true")
    args = parser.parse_args()
    bundle = build(skip_judges=args.skip_judges, target=args.target)
    if not args.build_only:
        demo(bundle, limit=args.limit, target_mode=args.target)
        if args.foundry_eval:
            foundry_eval(bundle, limit=args.limit)
    print_version_operations()

**Notebook users: skip the script-only entry-point cell above.** When running cell by cell in Jupyter or VS Code,
the earlier cells provide the notebook path. You do not need to call `main()`, `build()`, or `demo()` again.
The final code cell is only the command-line entry point for running the adjacent `.py` file as one program.
Its command-line invocation is guarded in this generated notebook; running the cell does not launch the lab.
For script mode instead, run `python <script>.py --help` in a Bash terminal from the lab folder and follow that lab's
CLI instructions. For notebook mode, follow the setup cell's kernel working-directory instructions.